In [4]:
import sys
sys.version

'3.12.0 | packaged by Anaconda, Inc. | (main, Oct  2 2023, 17:20:38) [MSC v.1916 64 bit (AMD64)]'

# Step 2: data collection

In [5]:
import pandas as pd 
df = pd.read_csv(r"data/raw/Cars24_Used_Cars_Raw_Data.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/Cars24_Used_Cars_Raw_Data.csv'

## Data Understanding 

#### Data Types

In [ ]:
df.info()

# Step 3 : EDA

#### missing values check 

In [ ]:
df.isna().sum()

### Duplicates records 

In [ ]:
df.duplicated().sum()

## Stats Features 

In [ ]:
df.describe()

In [ ]:
df.describe(include="all")

## value counts of key categorical features 

In [ ]:
df["Brand"].value_counts()

In [ ]:
df["City"].value_counts()

In [ ]:
df["Fuel_Type"].value_counts()

In [ ]:
df["Drive_Mode"].value_counts()

In [ ]:
df.columns

## Varaints 

In [ ]:
df["Variant"]

### 'Kilometer'

In [ ]:
df['Kilometer'] = df['Kilometer'].str.replace(" km", "").str.replace(",","").astype(int)

In [ ]:
min(df['Kilometer']), max(df['Kilometer'])

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns 

In [ ]:
df['Kilometer'].plot(kind="hist")

In [ ]:
df['Kilometer'].plot(kind="kde")

In [ ]:
sns.distplot(df["Kilometer"])

In [ ]:
df[["Kilometer"]].boxplot()

In [ ]:
df["Kilometer"].quantile(0.75)

In [ ]:
IQR = df["Kilometer"].quantile(0.75) - df["Kilometer"].quantile(0.25)
IQR

In [ ]:
upper_tail = df["Kilometer"].quantile(0.75)  + (1.5 * IQR) 
upper_tail

In [ ]:
df["Kilometer"].quantile(0.90)

In [ ]:
df["Kilometer"].quantile(0.95)

In [ ]:
df["Kilometer"].quantile(0.99)

In [ ]:
df[df["Kilometer"] <= 600000]

In [ ]:
df.shape

In [ ]:
3139 - 3073

In [ ]:
df[df["Kilometer"] <= 500000]

In [ ]:
3139 - 3032

In [ ]:
 df1 = df[df["Kilometer"] <= 500000]

In [6]:
df1[["Kilometer"]].boxplot()

NameError: name 'df1' is not defined

### observations 

In [ ]:
1. it contains lot of outliers
2. 90th percentile value is 95700 and rest 90th to 10th percentile values are huge. 
3. upper_tail from outlier also crossing this observations 
#### Conclusion 
as 95th perctile value is around 250000 so filtering the samples belwo 500000 


In [ ]:
 df = df[df["Kilometer"] <= 500000]

In [ ]:
df.columns

## Price 

In [ ]:
df["Price"].head()

In [ ]:
### observation 
1. it contains rupee symbol and lakhs as trailing text 
2. some samples contains two values 


In [ ]:
## after meeting those samples contain two values meaning one value is actual value and second discounted value. 
## we are going to use discounted price which is last one. 

# Step 4: Feature engineering 

### handling missing values 

In [ ]:
df.isna().sum()

In [ ]:
df[df["Fuel_Type"].isnull()]

In [ ]:
df.isna().sum() / df.shape[0] * 100

#### observation 

In [ ]:
1. All missing value were observed in toyota brand 
2. overall 0.41% values are missing which is small compared to total dataset
### conclusion 
3. rows/samples removed to maintain dataset original integrity. 

In [ ]:
df = df.dropna(subset=["Fuel_Type"])

In [ ]:
df.shape

# cleaning price columns 

In [ ]:
df["Price"]

In [ ]:
price_value = df["Price"].str.findall(r"\d+\.?\d*")
price_value

In [ ]:
float(price_value[0][-1]) * 100000

In [ ]:
df["Final_Price"] = price_value.apply(
    lambda x: float(x[-1]) * 100000
)

In [ ]:
df["Final_Price"]

In [ ]:
df

In [ ]:
df["Final_Price"].isna().sum()

In [ ]:
df["Final_Price"].info()

In [ ]:
df["Final_Price"].describe()

In [ ]:
df[["Final_Price"]].boxplot()

In [ ]:
## droping raw price columns 

In [ ]:
df.drop("Price", axis=1, inplace=True)

In [ ]:
df

# EMI

In [ ]:
df["EMI"]

In [ ]:
df["EMI"] = df["EMI"].str.extract(r"₹([\d,]+)")

In [ ]:
df["EMI"]

In [ ]:
df["EMI"] = df["EMI"].str.replace(",","")
df["EMI"]

In [ ]:
df["EMI"] = df["EMI"].astype(float)

In [ ]:
df["EMI"]

In [ ]:
df.info()

##  Fuel_Type

In [ ]:
df["Fuel_Type"].value_counts()

In [ ]:
df["Fuel_Type"].isna().sum()

## Drive_Mode   

In [ ]:
df["Drive_Mode"].value_counts()

### Client meeting 

In [ ]:
# car age 
# km_per_year 
# price_segment 
# age_bucket

In [ ]:
df

## car_age

In [ ]:
CURRENT_YEAR = 2026
df["Car_Age"] = CURRENT_YEAR - df["Model Year"] 

In [ ]:
df

## km_per_year 

In [ ]:
df["km_per_year"] = round(df["Kilometer"] / df["Car_Age"],1)
df["km_per_year"]

## # price_segment 

In [ ]:
vehicles are categorized into price ranges: 
budegt - upto 5L
mid - 5-10 
upper-mid - 10-20 
premium - 20L


In [ ]:
max(df["Final_Price"])

In [ ]:
bins = [0,500000,1000000,2000000,10000000]
labels = ["Budget","Mid","Upper-Mid","Premium"]
df["Price_Segment"] = pd.cut(df["Final_Price"], bins= bins, labels = labels)

In [ ]:
df["Price_Segment"].value_counts()

In [ ]:
df["Price_Segment"].value_counts()/df.shape[0] * 100

## age bucket 

In [ ]:
vechiels are grouped into age group 

0-3 year 
4-6 year 
7-10 year 
10+ year 

In [ ]:
bins = [0,3,6,10,20]
labels = ["0-3","4-6","7-10","10+"]
df["Age_Bucket"] = pd.cut(df["Car_Age"], bins= bins, labels= labels)

In [ ]:
df["Age_Bucket"].value_counts()

In [ ]:
df

# final clean dataset 

In [ ]:
df.to_csv(r"data/clean/car_clean.csv")

## Univariate Analysis 

### distribution of resale Price 

In [ ]:
df.columns

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns 
plt.figure(figsize=(10,7))
plt.hist(df["Final_Price"], bins=50, color="Green",edgecolor="black")
plt.title("Distribution of Price")
plt.xlabel("Price")
plt.ylabel("Number of cars")
plt.show()

In [ ]:
# observation 
1. price distribution is right -skewed 
2. most of card are price below 2 lakhs 

### kilometers

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns 
plt.figure(figsize=(12,5))
plt.hist(df["Kilometer"], bins=50, color="Orange",edgecolor="black")
plt.title("Distribution of Kilometer")
plt.xlabel("Kilometer")
plt.ylabel("Number of cars")
plt.show()

# car Age

In [ ]:
df.columns

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns 
plt.figure(figsize=(12,5))
plt.hist(df["Car_Age"], bins=20, color="Red",edgecolor="black")
plt.title("Distribution of car Age")
plt.xlabel("Car Age")
plt.ylabel("Number of cars")
plt.show()

## Bivariate Analysis

#### brand - wise resale prcie 

In [ ]:
df.columns

In [ ]:
df["Brand"].value_counts()

In [ ]:
df.groupby("Brand")["Final_Price"].mean()

In [ ]:
brand_avg_price = df.groupby("Brand")["Final_Price"].mean()
brand_avg_price = brand_avg_price.sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(12,5))
plt.bar(brand_avg_price.index,brand_avg_price.values, color="teal",edgecolor="black")
plt.title("Average resale price by brand ")
plt.xlabel("Brand")
plt.ylabel("Average Price")
plt.show()

In [ ]:
df.columns

## Average price by fuel type

In [ ]:
fuel_avg_price = df.groupby("Fuel_Type")["Final_Price"].mean()
fuel_avg_price = fuel_avg_price.sort_values(ascending=False)
plt.figure(figsize=(12,5))
plt.bar(fuel_avg_price.index,fuel_avg_price.values, color="coral",edgecolor="black")
plt.title("Average resale price by Fuel type ")
plt.xlabel("Fuel Type")
plt.ylabel("Average Price")
plt.show()

### Multivariate Analysis

#### brand-wise Price distribution 

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=df , x = "Brand", y = "Final_Price")
plt.show()

In [ ]:
df.columns

### price by Fuel Type and Transmission

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=df , x = "Fuel_Type", y = "Final_Price", hue="Drive_Mode")
plt.title("Price by Fuel type and Transmission ")
plt.xlabel("Fuel Type")
plt.ylabel("Price")
plt.show()


### correlation heatmap 

In [ ]:
df.columns

In [ ]:
df.info()



In [ ]:
numeric_cols = ['Model Year','Kilometer', 'EMI', 'Car_Age', 'km_per_year','Final_Price']
corr = df[numeric_cols].corr()
plt.figure(figsize=(12,1))
sns.heatmap(corr.tail(1),annot=True)

# Step 5: Fetaure Selection

In [ ]:
1. Filter methods
2. Wrapper method 
3. Emebeded method 

In [ ]:
df.columns

In [ ]:
df["Brand"].value_counts().shape

In [ ]:
df1 = df.copy()

In [ ]:
df1.shape

In [ ]:
encoded_df = pd.get_dummies(df1,columns = ["Brand"])

In [ ]:
encoded_df.shape

In [ ]:
df1.head()

In [ ]:
encoded_df.head()

In [ ]:
df["Variant"].nunique()

# Final Features 

In [ ]:
categorical_features = ['Brand','Variant','City','Fuel_Type', 'Drive_Mode']
Numerical_features = ['Model Year','Kilometer','Car_Age','km_per_year',]

# Step 6 : Model 

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X = df.drop("Final_Price", axis=1)
y = df["Final_Price"]

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42,)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num","passthrough",Numerical_features),
        ("cat",OneHotEncoder(handle_unknown='ignore',),categorical_features)
    ]
)

# Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
lr_model = LinearRegression()

lr_pipeline = Pipeline(
                [
                    ("preprocessor",preprocessor),
                    ("model",lr_model)
                ]
            )

lr_pipeline.fit(X_train,y_train)

# Step 7: Model Evaluation 

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
def model_eval(actual,perd):
    mae = mean_absolute_error(actual,perd)
    mse = mean_squared_error(actual,perd)
    r2 = r2_score(actual,perd)

    print(f"Mean Absolute Error = {mae}")
    print(f"Mean Sqaured Error = {mse}")
    print(f"R2 Score = {r2}")

    return mae,mse,r2

In [ ]:
# on testing data 
y_pred = lr_pipeline.predict(X_test)
model_eval(y_test,y_pred)

## on training data 
y_pred = lr_pipeline.predict(X_train)
model_eval(y_train,y_pred)

In [ ]:
ignore_list =  ['Creta S 1.6 PETROL', 'X1 sDrive20i xLine', 'i20 ERA 1.', 'CARENS LUXURY PLUS 1.4 PETROL 7 STR', 'Creta SX AT 1.6 DIESEL', 'Bolero 2WD 7 SEATER', 'i20 Active 1.2 SX', 'Endeavour 2.5L 4X2 AT', 'SONET HTX ANNIVERSARY EDITION 1.5 AT', 'SELTOS HTE 1.5 DIESEL', 'Captur RXT DIESEL DUAL TONE', 'X1 sDrive 20d Expedition', 'TAIGUN  GT PLUS 1.5 TSI MT', 'NEW SANTRO SPORTZ CNG', 'Swift VXI O', 'XUV500 W10 1.', 'Fiesta SXI 1.4 DIESEL', 'NEXON XZA PLUS SUNROOF DARK PETROL', 'EXTER S 1.2 CNG MT', 'ALCAZAR PLATINUM(O) 6STR 1.5 AT', 'Amaze 1.5L I-DTEC SX', 'CARENS Premium (O) 1.5 Petrol 7 STR', 'Creta SX (O) 1.6 PETROL', 'Curvv Accomplished S 1.2 Petrol 7DCA', 'CARENS LUXURY PLUS 1.4 PETROL DCT 7 STR', 'MARAZZO M2 7STR', 'XUV700 AX 5 D AT 7 STR', 'Elite i20 MAGNA 1.', 'TUV300 T6 PLUS', 'EXTER  SX 1.2 AMT', 'NEXON SMART+ 1.2 PETROL', '5 Series 530I M SPORT', 'XL6 ALPHA AT', 'Jazz 1.5L I-DTEC SV', 'Swift LDI', 'Swift Dzire ZXI', 'NEXON XZ PLUS (PREMIUM) KAZIRANGA PETROL', 'NEXON XZ PLUS (PREMIUM) PETROL', 'SONET X LINE 1.5 AT', 'YARIS V MT', 'Classic 1.4 TITANIUM DIESEL', 'Vitara Brezza ZXI', 'SCORPIO CLASSIC S11 7STR', 'Fortuner 2.8 4X2 AT', 'SELTOS GTX PLUS 1.5 DIESEL AT', 'Etios CROSS 1.5 V', 'XL6 ZETA AT', 'Safari XZ PLUS DARK EDITION', 'CARENS LUXURY PLUS 1.5 IMT PETROL', 'Dzire ZXI PLUS', 'TIAGO EV XT LONG RANGE', 'Duster 110 PS RXS MT DIESEL', 'IGNIS ALPHA 1.2 DUAL TONE', 'PUNCH ACCOMPLISHED MT CNG', 'Verna 1.6 VTVT SX O', 'FREESTYLE TITANIUM 1.2 PETROL', 'Jetta COMFORTLINE TDI', 'Swift VDI AMT', 'TIAGO NRG XZ AMT', 'TAIGUN GT PLUS 1.5 TSI DSG', 'Ecosport TITANIUM+ 1.0L ECOBOOST', 'Indica Vista VX QUADRAJET', 'XUV300 W4 1.5 DIESEL', 'Cross Polo HIGHLINE TDI', 'Etios Liva VX', 'NEXON XZ PETROL', 'SELTOS  HTE 1.5 PETROL MT', 'TIAGO NRG  NRG XZA CNG', '5 Series 530i M Sport', 'City 1.5L I-VTEC VX (O) MT', 'X7 xDrive 40i', 'FREESTYLE TITANIUM PLUS 1.5 DIESEL', 'AURA SX PLUS 1.2 AMT', 'Thar LX D 4WD AT CONVERTIBLE', 'ALTROZ XZ Plus (S) LUX', 'Swift ZXi Plus Dual Tone AMT', 'Verna 1.6 VTVT SX AT', 'Vento TRENDLINE DIESEL 1.', 'Polo COMFORTLINE 1.0L TSI AT', 'NEW I20 ASTA (O) 1.2 IVT', 'SONET HTK PLUS 1.5 IMT', 'XUV300 W8(O) 1.2 PETROL DUAL TONE', '5 Series 520D LUXURY LINE', 'S PRESSO VXI AMT', 'SELTOS HTE (O)1.5 Petrol MT', 'SONET GTX Plus 1.0 Turbo Petrol DCT Dual Tone', 'ALTROZ XM PETROL', 'Mobilio 1.5L I-DTEC S', 'BREZZA VXI SMART HYBRID', 'S PRESSO VXI (O)', 'YARIS VX MT', 'Camry 2.5L AT', 'KUV 100 NXT K2 D 6S', 'Verna 1.6 SX (O) CRDI MT', 'XUV500 W8 AWD', 'Verna FLUIDIC 1.6 CRDI SX AT', 'Xcent SX AT 1.2 (O)', 'TIGOR XZ PLUS CNG', 'ALTROZ XZA PLUS', 'New Elantra 2.0 SX(O) AT PETROL', 'Zest XE PETROL', 'Ertiga VXI AT', 'VENUE SX 1.0 (O) TURBO', 'MARAZZO M6 8 STR'] 
len(ignore_list)

In [ ]:
len(X_test)

In [ ]:
607-104

In [ ]:
104/607 *100

In [ ]:
104*.8

# Bias and variance 

In [ ]:
Bias >> high 
varaince >> low 

Underfit 

## Random Forest 

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf_pipeline = Pipeline(
                [
                    ("preprocessor",preprocessor),
                    ("model",rf)
                ]
            )

rf_pipeline.fit(X_train,y_train)

# on testing data 
y_pred = rf_pipeline.predict(X_test)
model_eval(y_test,y_pred)

## on training data 
y_pred = rf_pipeline.predict(X_train)
model_eval(y_train,y_pred)

### bias and variance 

In [ ]:
Bias >> low 
varaince >> high 

Overfit

## xgboost 

In [ ]:
from xgboost import XGBRegressor

In [ ]:
xgb = XGBRegressor(
    n_estimators = 500,
    learning_rate = 0.05,
    max_depth = 6,
    subsample = 0.8,
    colsample_bytree = 0.8, 
    objective = "reg:squarederror",
    random_state = 42
)

xgb_pipeline = Pipeline(
                [
                    ("preprocessor",preprocessor),
                    ("model",xgb)
                ]
            )

xgb_pipeline.fit(X_train,y_train)

# on testing data 
y_pred = xgb_pipeline.predict(X_test)
model_eval(y_test,y_pred)

## on training data 
y_pred = xgb_pipeline.predict(X_train)
model_eval(y_train,y_pred)

In [ ]:
Bias >> low 
varaince >> high 

Overfit

# Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV

In [ ]:
xg_hy_model = XGBRegressor(random_state=42)

pipeline = Pipeline(
                [
                    ("preprocessor",preprocessor),
                    ("model",xg_hy_model)
                ]
            )

hyp = {
    "model__n_estimators": [200,400,600,800],
    "model__learning_rate" :[0.01,0.07,0.1],
    "model__max_depth" :[4,5,7,8,9],
    "model__subsample" : [0.7,0.9,1.0]
}

rscv = RandomizedSearchCV(pipeline,hyp, random_state=42,n_jobs=-1)
rscv.fit(X_train,y_train)

In [ ]:
rscv.best_params_

In [ ]:
rscv.best_score_

In [ ]:
hyp_model = rscv.best_estimator_
# on testing data 
y_pred = hyp_model.predict(X_test)
model_eval(y_test,y_pred)

## on training data 
y_pred = hyp_model.predict(X_train)
model_eval(y_train,y_pred)

# Save the model 

In [ ]:
import os 
import pickle 

os.makedirs("model",exist_ok=True)
with open("model/xgb_model.pkl","wb") as model: 
    pickle.dump(xgb_pipeline,model)

In [ ]:
X_train.head(1).to_numpy()

In [ ]:
X_train.columns

In [ ]:
dict(zip(X_train.columns,X_train.head(1).to_numpy()[0]))

In [ ]:
X_train.iloc[3]

In [ ]:
X_train["Brand"].value_counts()

In [ ]:
X_train.iloc[5]